In [2]:
import pandas as pd

data = pd.read_csv(r"C:\Users\User\Downloads\Synthetic_Clinical_Data.csv")
print(data)

print(data.head())

   Patient_ID Enrollment_Date Visit_Type  Visit_Date  Age Gender  Systolic_BP  \
0       PT001      15-01-2026   Baseline  15-01-2026   55      M        145.0   
1       PT001      15-01-2026     Week 4  12-02-2026   55      M        138.0   
2       PT001      15-01-2026     Week 8  12-03-2026   55      M        132.0   
3       PT002      16-01-2026   Baseline  16-01-2026   62      F        158.0   
4       PT002      16-01-2026     Week 4  13-02-2026   62      F          NaN   
5       PT002      16-01-2026     Week 8  13-03-2026   62      F        145.0   
6       PT003      17-01-2026   Baseline  17-01-2026   48      M        135.0   
7       PT003      17-01-2026     Week 4  14-02-2026   48      M        130.0   
8       PT003      17-01-2026     Week 8  14-03-2026   48      M        125.0   
9       PT004      18-01-2026   Baseline  18-01-2026   71      M        168.0   
10      PT004      18-01-2026     Week 4  15-02-2026   71      M        155.0   
11      PT004      18-01-202

In [3]:
print("Total Records:", len(data))
print("Total Variables:", len(data.columns))

print("\nFirst 5 Records:")
print(data.head())

Total Records: 60
Total Variables: 18

First 5 Records:
  Patient_ID Enrollment_Date Visit_Type  Visit_Date  Age Gender  Systolic_BP  \
0      PT001      15-01-2026   Baseline  15-01-2026   55      M        145.0   
1      PT001      15-01-2026     Week 4  12-02-2026   55      M        138.0   
2      PT001      15-01-2026     Week 8  12-03-2026   55      M        132.0   
3      PT002      16-01-2026   Baseline  16-01-2026   62      F        158.0   
4      PT002      16-01-2026     Week 4  13-02-2026   62      F          NaN   

   Diastolic_BP  Heart_Rate  Weight_kg Medication_Name  Dose_mg  \
0            92          72       78.5      Lisinopril     10.0   
1            88          70       78.2      Lisinopril     10.0   
2            85          68       77.9      Lisinopril     10.0   
3            98          76       82.1       Enalapril      5.0   
4            96          74       82.0       Enalapril      5.0   

   Compliance_Percent  Fasting_Glucose_mg_dL  Total_Choleste

In [4]:
# Check missing values

missing_values = data.isnull().sum()

print("\nMISSING VALUES")

print(missing_values[missing_values > 0])


MISSING VALUES
Systolic_BP                 1
Fasting_Glucose_mg_dL       1
Total_Cholesterol_mg_dL     1
Adverse_Event              47
AE_Severity                47
dtype: int64


In [5]:
# Store all data issues here

issues = []

In [6]:
# Check missing Systolic BP

missing_bp = data[data["Systolic_BP"].isnull()]

for index, row in missing_bp.iterrows():

    issues.append([
        row["Patient_ID"],
        row["Visit_Type"],
        "Systolic_BP",
        "Missing value",
        "High",
        "Open"
    ])

In [7]:
# Check Age range

invalid_age = data[
    (data["Age"] < 18) |
    (data["Age"] > 100)
]

for index, row in invalid_age.iterrows():

    issues.append([
        row["Patient_ID"],
        row["Visit_Type"],
        "Age",
        "Age outside valid range",
        "High",
        "Open"
    ])

In [8]:
# Check Systolic BP range

invalid_systolic = data[
    (data["Systolic_BP"] < 90) |
    (data["Systolic_BP"] > 220)
]

for index, row in invalid_systolic.iterrows():

    issues.append([
        row["Patient_ID"],
        row["Visit_Type"],
        "Systolic_BP",
        "Systolic BP outside valid range",
        "Medium",
        "Open"
    ])

In [9]:
# Check Diastolic BP range

invalid_diastolic = data[
    (data["Diastolic_BP"] < 60) |
    (data["Diastolic_BP"] > 120)
]

for index, row in invalid_diastolic.iterrows():

    issues.append([
        row["Patient_ID"],
        row["Visit_Type"],
        "Diastolic_BP",
        "Diastolic BP outside valid range",
        "Medium",
        "Open"
    ])

In [10]:
# Check Blood Pressure logic

bp_error = data[
    data["Systolic_BP"] <= data["Diastolic_BP"]
]

for index, row in bp_error.iterrows():

    issues.append([
        row["Patient_ID"],
        row["Visit_Type"],
        "Blood Pressure",
        "Systolic BP should be greater than Diastolic BP",
        "High",
        "Open"
    ])

In [11]:
# Create Edit Clarification Log

issues_df = pd.DataFrame(
    issues,
    columns=[
        "Patient_ID",
        "Visit",
        "Variable",
        "Issue",
        "Severity",
        "Status"
    ]
)

issues_df.to_csv(
    "outputs/Edit_Clarification_Log.csv",
    index=False
)

print("\nTotal Issues:", len(issues_df))
print("Edit Clarification Log created")


Total Issues: 1
Edit Clarification Log created


In [12]:
# Calculate Data Completeness

total_values = data.size

missing_count = data.isnull().sum().sum()

completeness = (
    (total_values - missing_count) / total_values
) * 100

print("\nDATA COMPLETENESS")

print(
    "Data Completeness:",
    round(completeness, 2),
    "%"
)


DATA COMPLETENESS
Data Completeness: 91.02 %


In [13]:
# Descriptive Statistics

print("\nDESCRIPTIVE STATISTICS")

numeric_columns = [
    "Age",
    "Systolic_BP",
    "Diastolic_BP",
    "Heart_Rate",
    "Weight_kg"
]

print(
    data[numeric_columns].describe()
)


DESCRIPTIVE STATISTICS
             Age  Systolic_BP  Diastolic_BP  Heart_Rate  Weight_kg
count  60.000000    59.000000     60.000000   60.000000  60.000000
mean   59.650000   145.881356     90.916667   73.083333  82.780000
std     6.751836    11.107756      7.033315    4.927583   5.132509
min    48.000000   125.000000     79.000000   64.000000  75.300000
25%    54.750000   138.000000     85.000000   70.000000  78.500000
50%    58.500000   145.000000     90.000000   72.000000  82.050000
75%    64.750000   153.500000     96.000000   76.250000  87.000000
max    71.000000   172.000000    108.000000   85.000000  92.700000


In [14]:
# Create Data Quality Report

with open(
    "outputs/CDM_Data_Quality_Report.txt",
    "w"
) as file:

    file.write(
        "CLINICAL DATA MANAGEMENT REPORT\n"
    )

    file.write(
        "================================\n\n"
    )

    file.write(
        "Total Records: "
        + str(len(data))
        + "\n"
    )

    file.write(
        "Total Variables: "
        + str(len(data.columns))
        + "\n"
    )

    file.write(
        "Data Completeness: "
        + str(round(completeness, 2))
        + "%\n"
    )

    file.write(
        "Total Issues Found: "
        + str(len(issues_df))
        + "\n"
    )

print("\nData Quality Report created")


Data Quality Report created


In [15]:
# Project completed

print("\nDATA REVIEW COMPLETED")

print("\nOutput Files Created:")

print("1. Edit_Clarification_Log.csv")

print("2. CDM_Data_Quality_Report.txt")


DATA REVIEW COMPLETED

Output Files Created:
1. Edit_Clarification_Log.csv
2. CDM_Data_Quality_Report.txt
